# S2 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa

**Actividad:** construir el notebook `02_fundamentos_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), aplicando extracción, transformaciones, funciones, agrupaciones/agregaciones, RDD y verificando en cada paso el efecto de la evaluación perezosa — sobre el dataset real H&M (Kaggle).

**Propósito de la actividad:** dejar evidencia ejecutable de que dominas el ciclo completo de transformación distribuida en PySpark — DataFrame y RDD sobre la misma `SparkSession` — antes de avanzar a formatos analíticos particionados (S3) y ML distribuido (S4).

Guía completa: `docs/sesiones/S02_Fundamentos_PySpark_Transformaciones_Lazy_Evaluation.md`, sección 3 (pasos 3.1 a 3.11).

## 3.1 Descargar el dataset H&M y reanudar el entorno `lambda26`

**Producto del paso:** dataset H&M disponible en `pyspark/sesiones/s02-fundamentos/data/`, entorno `lambda26` funcionando.

El dataset ya está descargado en `data/` (`articles.csv`, `customers.csv`, `transactions.parquet`). Si el contenedor `lambda26` sigue corriendo desde S1, continúa directo en 3.2.

## 3.2 Crear el notebook y la `SparkSession`

**Producto del paso:** notebook con una `SparkSession` activa.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 22:25:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


`spark.driver.memory` en `"4g"`: en modo `local[*]` el driver y el executor comparten un mismo proceso JVM, y sin fijar esto Spark usa el default de 1g — sin importar cuánta RAM tenga el servidor. Con varias lecturas/escrituras y agrupaciones encadenadas en un mismo notebook, 1g se queda corto y la JVM puede caerse (se ve como `ConnectionRefusedError` del lado de Python, aunque el servidor tenga memoria de sobra sin usar). Si el notebook se cae igual, reinicia el kernel y sube este valor (`"8g"`, `"16g"`, según la RAM disponible).

Declara la ruta del dataset como variable global, una sola vez — úsala en cada lectura del resto del notebook.

In [2]:
ORIGEN_DATOS = "/opt/s02-fundamentos/data"
ARTIFACTS = "/opt/s02-fundamentos/artifacts"

## 3.3 Cargar y explorar `articles.csv`

**Producto del paso:** DataFrame `df_articles` cargado (primera forma de lectura: `inferSchema`), con estructura, filas y estadísticas verificadas paso a paso.

Primero, la lectura:

In [3]:
df_articles = spark.read.csv(
    f"{ORIGEN_DATOS}/articles.csv",
    header=True,
    inferSchema=True,
)

`.show()` — visualiza filas en formato tabla:

In [4]:
df_articles.show(5, truncate=False)

+----------+------------+-----------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+----------------------+----------------+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|article_id|product_code|prod_name        |product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|

`.printSchema()` — muestra el esquema (nombres, tipos, nulabilidad):

In [5]:
df_articles.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

`.describe()` — resumen estadístico. Con las 25 columnas de `articles.csv`, ni siquiera `vertical=True` lo deja cómodo (125 líneas) — mejor selecciona antes un puñado de columnas representativas, mezclando nominales y numéricas:

In [6]:
df_articles.select(
    "prod_name", "product_group_name", "colour_group_name", "department_no", "section_no"
).describe().show()

26/08/20 22:25:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:======>                                                    (1 + 8) / 9]

+-------+--------------------+------------------+-----------------+------------------+-----------------+
|summary|           prod_name|product_group_name|colour_group_name|     department_no|       section_no|
+-------+--------------------+------------------+-----------------+------------------+-----------------+
|  count|              105542|            105542|           105542|            105542|           105542|
|   mean|                NULL|              NULL|             NULL|  4532.77783252165|42.66421898391162|
| stddev|                NULL|              NULL|             NULL|2712.6920114304908|23.26010495890632|
|    min|& Denim Boyfriend...|       Accessories|            Beige|              1201|                2|
|    max|           Åsa Dress|           Unknown|  Yellowish Brown|              9989|               97|
+-------+--------------------+------------------+-----------------+------------------+-----------------+



Parámetros de `.show()`: `vertical=True` muestra cada fila como lista de campos, útil con muchas columnas.

In [7]:
df_articles.show(3, vertical=True)

-RECORD 0--------------------------------------------
 article_id                   | 108775015            
 product_code                 | 108775               
 prod_name                    | Strap top            
 product_type_no              | 253                  
 product_type_name            | Vest top             
 product_group_name           | Garment Upper body   
 graphical_appearance_no      | 1010016              
 graphical_appearance_name    | Solid                
 colour_group_code            | 9                    
 colour_group_name            | Black                
 perceived_colour_value_id    | 4                    
 perceived_colour_value_name  | Dark                 
 perceived_colour_master_id   | 5                    
 perceived_colour_master_name | Black                
 department_no                | 1676                 
 department_name              | Jersey Basic         
 index_code                   | A                    
 index_name                 

## 3.4 Cargar `customers.csv` con esquema explícito y explorar columnas

**Producto del paso:** DataFrame `df_customers` cargado con `StructType` (segunda forma de lectura), con sus columnas y su tamaño verificados.

Define el esquema, columna por columna:

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

Lee el CSV con ese esquema:

In [9]:
df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers,
)

Resumen estadístico:

In [10]:
df_customers.describe().show()

[Stage 7:=>                                                       (1 + 49) / 50]

+-------+--------------------+------+------+------------------+----------------------+------------------+--------------------+
|summary|         customer_id|    FN|Active|club_member_status|fashion_news_frequency|               age|         postal_code|
+-------+--------------------+------+------+------------------+----------------------+------------------+--------------------+
|  count|             1371980|476930|464404|           1365918|               1355971|           1356119|             1371980|
|   mean|                NULL|   1.0|   1.0|              NULL|                  NULL|   36.386964565794|                NULL|
| stddev|                NULL|   0.0|   0.0|              NULL|                  NULL|14.313627981628082|                NULL|
|    min|00000dbacae5abe5e...|   1.0|   1.0|            ACTIVE|               Monthly|                16|0000198d2c593b7d3...|
|    max|ffffd9ac14e899464...|   1.0|   1.0|        PRE-CREATE|             Regularly|                99|ffffe1

Fíjate en la fila `count` de cada columna: en una corrida real, `customer_id` da 1 371 980 (todas las filas), pero `FN` y `Active` dan bastante menos — cerca de dos tercios de los valores son nulos en esas dos columnas. No es un error de lectura: así llega el dato real de Kaggle. Profundizar en por qué y qué hacer con esos nulos es justamente lo que S3 formaliza como control de calidad de datos — acá basta con que lo notes.

Nombres de columna:

In [11]:
print(df_customers.columns)

['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']


Cantidad de filas y de columnas:

In [12]:
num_rows, num_cols = df_customers.count(), len(df_customers.columns)
print(f"Filas: {num_rows}, Columnas: {num_cols}")

Filas: 1371980, Columnas: 7


**Muestra aleatoria** (`.sample()`): con 1 371 980 filas (confírmalo con `.count()` arriba), trabajar con todo el dataset en una laptop se vuelve pesado — para explorar y validar la lógica alcanza con una muestra.

In [13]:
df_customers_muestra = df_customers.sample(
    withReplacement=False,  # sin reemplazo: cada fila se elige como máximo una vez
    fraction=0.01,          # ~1% del dataset
    seed=None,              # sin semilla fija: cada corrida da una muestra distinta
)

**Registros iniciales** (`.head()`):

In [14]:
df_customers_muestra.head(3)

[Row(customer_id='0000b7a134c3ec0d8842fad1fd4ca28517424c14fc48485d65bd332309b7439a', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=75, postal_code='e3f4042e6f6c5d8165f7c08fc865eb7e9281396ddca498a394817d9d918646f6'),
 Row(customer_id='000fe0cb78750fe7a65aa52943df29b6b781016fda241685fb814e2fc0f427a1', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=21, postal_code='000a3543b8ec684af4b8ee0e730304d2a814646202117263899f05fd1c526bc7'),
 Row(customer_id='0012d60b384f6e693d4341629e5dc7211051800aaaf452dcdf9419ab5ab38a08', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=56, postal_code='cdc5af66ea6ff8d918c6d71c77563ce5afc7ff0c877a03d719af65dca6db781d')]

**Registros finales** (`.tail()`):

In [15]:
df_customers_muestra.tail(3)

[Row(customer_id='fffb04448b4082028fcf3f205b05f0ee846da598eeca1f70fadd51494224af42', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=53, postal_code='ddecb6c158a23f126fc9fbf0e361a4697f54875e96a55f49768c1eb10f8e0395'),
 Row(customer_id='fffcb073cfbea83431228195820e21c82b56905b3ee1ebfcdb02c838df287258', FN=1.0, Active=1.0, club_member_status='PRE-CREATE', fashion_news_frequency='Regularly', age=56, postal_code='68b87dcf867c4df025400d295929c8faa183ded44efdac47483a19527f37eaf3'),
 Row(customer_id='fffcd556af797bddc25d6d56600b6e298a19b90624a8eee515ef774c0584c2a5', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=21, postal_code='25c625af5e69ad37fd081e0beae98938cf4598a90aed9807b6f7e66783111448')]

**Guardar la muestra en CSV y Parquet:** adelanto de lo que S3 formaliza a fondo — por ahora, solo la mecánica básica de escribir un resultado a disco. Spark escribe una **carpeta** (un archivo por partición adentro), no un solo archivo; `mode("overwrite")` reemplaza la carpeta si ya existe.

In [16]:
df_customers_muestra.write.mode("overwrite").csv(f"{ARTIFACTS}/customers_muestra_csv", header=True)
df_customers_muestra.write.mode("overwrite").parquet(f"{ARTIFACTS}/customers_muestra_parquet")

26/08/20 22:25:53 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 98.06% for 31 writers
26/08/20 22:25:53 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 95.00% for 32 writers
26/08/20 22:25:53 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 92.12% for 33 writers
26/08/20 22:25:53 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 89.41% for 34 writers
26/08/20 22:25:53 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 86.86% for 35 writers
26/08/20 22:25:53 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 84.44% for 36 writers
26/08/20 22:25:53 WARN MemoryManager: Total allocation exceeds 9

**¿Cuántos archivos quedaron?** `.rdd.getNumPartitions()` dice cuántas particiones tiene el DataFrame — y por lo tanto, cuántos `part-0000X-...` escribe. En una corrida real dio **50**:

In [17]:
df_customers_muestra.rdd.getNumPartitions()

50

`df_customers_muestra` hereda el número de particiones de la lectura original de `customers.csv` (no se reduce solo porque `.sample()` deje menos filas) — por eso salen varias decenas de `part-0000X-...` para una muestra que en realidad es chica.

**Para compartir un solo archivo** (por ejemplo con estudiantes cuya laptop tiene pocos recursos, en vez de una carpeta con ~20 partes): junta las particiones en una sola con `.coalesce(1)` antes de escribir:

In [18]:
df_customers_muestra.coalesce(1).write.mode("overwrite").csv(f"{ARTIFACTS}/customers_muestra_csv_unico", header=True)
df_customers_muestra.coalesce(1).write.mode("overwrite").parquet(f"{ARTIFACTS}/customers_muestra_parquet_unico")

Esto sigue creando una carpeta (con un único `part-00000-...` adentro, más `_SUCCESS`) — el archivo real a compartir es ese `part-00000-...`; puedes renombrarlo o descargarlo directo desde el explorador de archivos de Jupyter.

**¿Y si quiero un `.csv` de verdad, con nombre exacto, sin carpeta?** Spark no lo hace — ni siquiera `.coalesce(1)` cambia eso (sigue generando `part-00000-...` dentro de una carpeta). Como la muestra ya es chica (~1%, cabe en memoria), conviértela a pandas y usa su `.to_csv()`, que sí escribe un único archivo con el nombre exacto:

In [19]:
df_customers_muestra.toPandas().to_csv(f"{ARTIFACTS}/customers_muestra.csv", index=False)

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


`index=False` evita que pandas agregue una columna extra con el número de fila. Esta ruta solo es segura para datos que caben en memoria del driver — para el dataset completo (millones de filas) seguiría siendo `.coalesce(1)` + `.write.csv()` de Spark, no `.toPandas()`.

**Leer de vuelta, tenga uno o varios archivos:** Spark lee la carpeta completa como un solo DataFrame — no importa si adentro hay un `part-00000` o veinte, la lectura es igual de simple:

In [20]:
df_leido_csv = spark.read.csv(f"{ARTIFACTS}/customers_muestra_csv", header=True, inferSchema=True)
df_leido_parquet = spark.read.parquet(f"{ARTIFACTS}/customers_muestra_parquet")

df_leido_csv.count(), df_leido_parquet.count()

(13678, 13678)

## 3.5 Aplicar transformaciones y verificar la evaluación perezosa

**Producto del paso:** evidencia de que el plan se construye antes de ejecutarse.

Primero, `.select()` — elige columnas específicas del DataFrame:

In [21]:
df_seleccionado = df_customers.select("customer_id", "age", "club_member_status", "fashion_news_frequency")

Ahora, `.filter()` — conserva solo las filas que cumplen una condición:

In [22]:
from pyspark.sql.functions import col

df_activos = df_seleccionado.filter(col("club_member_status") == "ACTIVE")

Como práctica, combina ambas en una sola expresión encadenada:

In [23]:
df_activos = (
    df_customers
    .select("customer_id", "age", "club_member_status", "fashion_news_frequency")
    .filter(col("club_member_status") == "ACTIVE")
)

# Hasta aquí Spark solo construyó el plan: no hay salida, no hubo ejecución
df_activos

DataFrame[customer_id: string, age: int, club_member_status: string, fashion_news_frequency: string]

Acción: aquí recién Spark ejecuta.

In [24]:
df_activos.show(10, truncate=False)
df_activos.count()

+----------------------------------------------------------------+---+------------------+----------------------+
|customer_id                                                     |age|club_member_status|fashion_news_frequency|
+----------------------------------------------------------------+---+------------------+----------------------+
|00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657|49 |ACTIVE            |NONE                  |
|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|25 |ACTIVE            |NONE                  |
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|24 |ACTIVE            |NONE                  |
|00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e|54 |ACTIVE            |NONE                  |
|00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a|52 |ACTIVE            |Regularly             |
|0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d8cd0c725276a467a2a|20 |ACTIVE            |NONE   

1272491

## 3.6 Analizar el plan de ejecución con `explain()`

**Producto del paso:** plan de ejecución interpretado con al menos una optimización identificada.

Usamos `explain()` para observar cómo Spark organiza el procesamiento antes de ejecutarlo — es la base para entender rendimiento y optimización, no solo curiosidad interna: el mismo comando te sirve más adelante para diagnosticar por qué una consulta es lenta o si Catalyst está aplicando las optimizaciones que esperás (predicate pushdown, column pruning).

In [25]:
df_activos.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('club_member_status, ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Analyzed Logical Plan ==
customer_id: string, age: int, club_member_status: string, fashion_news_frequency: string
Filter (club_member_status#600 = ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Optimized Logical Plan ==
Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
+- Filter (isnotnull(club_member_status#600) AND (club_member_status#600 = ACTIVE))
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== 

**Cómo leer esto, plan por plan.** Cada plan se lee de **abajo hacia arriba** (lo de más abajo es lo primero que pasa):

- **`Parsed Logical Plan`:** tu código tal cual lo escribiste, sin revisar nada todavía. De abajo hacia arriba: `Relation ... csv` (lee el archivo) → `Project [...]` (tu `select()`) → `Filter ...` (tu `filter()`) — el mismo orden en que escribiste `.select().filter()`. Las comillas simples (`'Filter`, `` `club_member_status` ``) significan "todavía no verificado": Spark ni confirmó que esas columnas existan.
- **`Analyzed Logical Plan`:** misma estructura, ninguna línea cambia de lugar. Spark solo confirmó que las columnas existen y les asignó tipo (`age: int`, `club_member_status: string`) — es un chequeo, no una optimización.
- **`Optimized Logical Plan`:** acá sí cambia algo — compáralo contra el *Parsed*. `Filter` y `Project` **se intercambian de posición**: ahora el filtro va justo después de leer el archivo, y el `select()` al final. Tu código dice "selecciona, después filtra"; Catalyst ejecuta "filtra primero, selecciona después", porque descartar filas cuanto antes reduce lo que el resto del plan tiene que procesar — es *predicate pushdown*. Además aparece `isnotnull(club_member_status)`, que tú nunca escribiste: Catalyst lo agrega solo, porque una igualdad (`= ACTIVE`) nunca es cierta para un valor nulo, así que de paso descarta los nulos.
- **`Physical Plan`:** la receta final de ejecución. `PushedFilters` confirma que el filtro se empujó hasta el propio lector del CSV; `ReadSchema` muestra que solo se leen las columnas que realmente usaste (4 de las 7 de `customers.csv`) — las demás ni se tocan.

**La idea en una frase:** tu código pide "trae estas columnas, después filtra"; Spark ejecuta "filtra (leyendo solo esas columnas) primero, presenta después" — mismo resultado, procesando muchísimo menos dato en el camino.

## 3.7 Aplicar funciones y crear columnas con `withColumn()`

**Producto del paso:** `df_articles` (cargado en 3.3) con al menos tres columnas nuevas o corregidas.

In [26]:
from pyspark.sql.functions import col, when, lit, current_date

**Corregir el tipo de una columna** (`.cast()`): `article_id` son solo dígitos, `inferSchema` corre el riesgo de quitarle el cero inicial.

In [27]:
df_articles = df_articles.withColumn("article_id", col("article_id").cast("string"))

**Clasificar con `when()`/`otherwise()`** (valores reales de `perceived_colour_value_name`: `"Dark"`, `"Light"`, u otros):

In [28]:
df_articles = df_articles.withColumn(
    "rango_percibido",
    when(col("perceived_colour_value_name") == "Dark", "oscuro")
    .when(col("perceived_colour_value_name") == "Light", "claro")
    .otherwise("medio")
)

**Agregar una columna constante** (`lit()`):

In [29]:
df_articles = df_articles.withColumn("fuente", lit("H&M Kaggle"))

**Agregar una columna con fecha actual** (`current_date()` — la calcula Spark, no tú):

In [30]:
df_articles = df_articles.withColumn("fecha_procesado", current_date())

Verifica el resultado — confirma que `article_id` conserva el cero inicial (ej. `0108775015`, no `108775015`):

In [31]:
df_articles.select(
    "article_id", "prod_name", "perceived_colour_value_name", "rango_percibido", "fuente", "fecha_procesado"
).show(5, truncate=False)

+----------+-----------------+---------------------------+---------------+----------+---------------+
|article_id|prod_name        |perceived_colour_value_name|rango_percibido|fuente    |fecha_procesado|
+----------+-----------------+---------------------------+---------------+----------+---------------+
|108775015 |Strap top        |Dark                       |oscuro         |H&M Kaggle|2026-08-20     |
|108775044 |Strap top        |Light                      |claro          |H&M Kaggle|2026-08-20     |
|108775051 |Strap top (1)    |Dusty Light                |medio          |H&M Kaggle|2026-08-20     |
|110065001 |OP T-shirt (Idro)|Dark                       |oscuro         |H&M Kaggle|2026-08-20     |
|110065002 |OP T-shirt (Idro)|Light                      |claro          |H&M Kaggle|2026-08-20     |
+----------+-----------------+---------------------------+---------------+----------+---------------+
only showing top 5 rows


## 3.8 Cargar `transactions.parquet` y aplicar funciones

**Producto del paso:** DataFrame `df_transactions` cargado (tercera forma de lectura: Parquet), con tipos corregidos y columnas clasificadas para poder agrupar en 3.9.

Primero, la lectura — trabajamos solo con un subconjunto (`.limit(100000)`) del archivo completo, para explorar sin procesar todo el volumen disponible:

In [32]:
df_transactions = spark.read.parquet(f"{ORIGEN_DATOS}/transactions.parquet").limit(100000)

Confirma tú mismo las columnas — no asumas el esquema:

In [33]:
from pyspark.sql.functions import col, to_date

df_transactions = (
    df_transactions
    .withColumn("t_dat", to_date(col("t_dat"), "yyyy-MM-dd"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("article_id", col("article_id").cast("string"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("sales_channel_id", col("sales_channel_id").cast("int"))
)

**Clasificar por canal de venta:**

In [34]:
from pyspark.sql.functions import when

df_transactions = df_transactions.withColumn(
    "canal",
    when(col("sales_channel_id") == 1, "Online")
    .when(col("sales_channel_id") == 2, "Tienda")
    .otherwise("Desconocido")
)

**Etiquetar transacciones baratas o caras:** recuerda que `price` está normalizado a [0, 1] — los umbrales van en esa escala, no en soles/dólares:

In [35]:
df_transactions = df_transactions.withColumn(
    "categoria_precio",
    when(col("price") < 0.1, "Barato")
    .when(col("price") < 0.3, "Medio")
    .otherwise("Caro")
)

**Aplicar múltiples condiciones** (con `&`, combinando dos columnas):

In [36]:
df_transactions = df_transactions.withColumn(
    "tipo_transaccion",
    when((col("sales_channel_id") == 1) & (col("price") > 0.3), "Online Premium")
    .when((col("sales_channel_id") == 2) & (col("price") > 0.3), "Tienda Premium")
    .otherwise("Regular")
)

**Fin de semana vs. laborable** (`dayofweek()` + `.isin()` — 1=domingo...7=sábado, convención propia de Spark, no ISO):

In [37]:
from pyspark.sql.functions import dayofweek

df_transactions = df_transactions.withColumn("dia_semana", dayofweek(col("t_dat")))

df_transactions = df_transactions.withColumn(
    "tipo_dia",
    when(col("dia_semana").isin(1, 7), "Fin de semana").otherwise("Laborable")
)

**Por qué no `date_format(col, "u")`:** una versión anterior de este notebook usaba `date_format(col("t_dat"), "u")`. Es un patrón de texto, y desde Spark 3.0 el parser de fechas cambió de `SimpleDateFormat` a `DateTimeFormatter` — en el parser nuevo, "u" significa **año**, no día de la semana; en el viejo significaba día de la semana. Como el significado cambió entre versiones, Spark no lo ejecuta en silencio: lanza `SparkUpgradeException` para forzar una decisión explícita. `dayofweek()` evita el problema porque no depende de un patrón de texto ambiguo.

**Por qué el error apareció en la celda de la función ventana (3.9) y no aquí:** evaluación perezosa. Esta celda solo arma el plan (`withColumn()` es transformación, no acción) — el error se dispara recién en la primera `.show()`/`.count()` que fuerza a evaluar toda la cadena de `df_transactions`, sin importar en qué celda esté esa acción.

## 3.9 Aplicar agrupaciones y agregaciones (`transactions.parquet`)

**Producto del paso:** resumen agregado por cliente, con la advertencia de dominio sobre `price` aplicada.

Todos los `.show()` de esta sección usan `truncate=False`: `customer_id` es un hash largo, y con el truncado por defecto (20 caracteres) dos clientes distintos pueden lucir idénticos en pantalla — sin `truncate=False` no podrías confirmar si dos filas son del mismo cliente o de dos distintos.

**Total gastado por cliente** (`sum`):

In [38]:
from pyspark.sql.functions import sum

df_total_por_cliente = df_transactions.groupBy("customer_id").agg(
    sum("price").alias("total_normalizado")
)
df_total_por_cliente.show(5, truncate=False)

[Stage 36:============================>                          (33 + 31) / 64]

+----------------------------------------------------------------+------------------+
|customer_id                                                     |total_normalizado |
+----------------------------------------------------------------+------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|0.081322033898305 |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|0.0863559322033897|
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|0.190593220338983 |
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|0.0428474576271185|
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|0.7130508474576257|
+----------------------------------------------------------------+------------------+
only showing top 5 rows


**Promedio de gasto por cliente** (`avg`):

In [39]:
from pyspark.sql.functions import avg

df_avg_por_cliente = df_transactions.groupBy("customer_id").agg(
    avg("price").alias("promedio_normalizado")
)
df_avg_por_cliente.show(5, truncate=False)

+----------------------------------------------------------------+--------------------+
|customer_id                                                     |promedio_normalizado|
+----------------------------------------------------------------+--------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|0.0406610169491525  |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|0.01727118644067794 |
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|0.038118644067796595|
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|0.02142372881355925 |
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|0.023768361581920857|
+----------------------------------------------------------------+--------------------+
only showing top 5 rows


**Número de transacciones por cliente** (`count`):

In [40]:
from pyspark.sql.functions import count

df_count_por_cliente = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones")
)
df_count_por_cliente.show(5, truncate=False)

+----------------------------------------------------------------+-----------------+
|customer_id                                                     |num_transacciones|
+----------------------------------------------------------------+-----------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|2                |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|5                |
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|5                |
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|2                |
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|30               |
+----------------------------------------------------------------+-----------------+
only showing top 5 rows


**Clientes únicos por día** (`countDistinct`):

In [41]:
from pyspark.sql.functions import countDistinct

df_clientes_unicos_por_dia = df_transactions.groupBy("t_dat").agg(
    countDistinct("customer_id").alias("clientes_unicos")
)
df_clientes_unicos_por_dia.show(5)

+----------+---------------+
|     t_dat|clientes_unicos|
+----------+---------------+
|2018-09-20|          13987|
|2018-09-21|          13932|
|2018-09-22|           1285|
+----------+---------------+



**Varias agregaciones a la vez** (más eficiente que calcularlas por separado):

In [42]:
from pyspark.sql.functions import sum, count, avg

df_agg_multi = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones"),
    sum("price").alias("total_normalizado"),
    avg("price").alias("promedio_normalizado")
)
df_agg_multi.show(5, truncate=False)

+----------------------------------------------------------------+-----------------+------------------+--------------------+
|customer_id                                                     |num_transacciones|total_normalizado |promedio_normalizado|
+----------------------------------------------------------------+-----------------+------------------+--------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|2                |0.081322033898305 |0.0406610169491525  |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|5                |0.0863559322033897|0.01727118644067794 |
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|5                |0.190593220338983 |0.038118644067796595|
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|2                |0.0428474576271185|0.02142372881355925 |
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|30               |0.7130508474576257|0.023768361581920857|


**Agrupar por múltiples columnas:**

In [43]:
df_ventas_por_dia_y_canal = df_transactions.groupBy("t_dat", "sales_channel_id").agg(
    sum("price").alias("total_normalizado")
)
df_ventas_por_dia_y_canal.show(5)

+----------+----------------+------------------+
|     t_dat|sales_channel_id| total_normalizado|
+----------+----------------+------------------+
|2018-09-20|               2| 1072.852593220397|
|2018-09-20|               1|342.49325423728436|
|2018-09-21|               2|1036.5931525424558|
|2018-09-21|               1| 382.6691525423702|
|2018-09-22|               1|55.550881355931914|
+----------+----------------+------------------+
only showing top 5 rows


**Usar `agg()` con un diccionario** (forma alternativa, sin `alias()`):

In [44]:
df_agg_dict = (
    df_transactions.groupBy("customer_id")
    .agg({"price": "sum", "article_id": "count"})
    .withColumnRenamed("sum(price)", "total_normalizado")
    .withColumnRenamed("count(article_id)", "num_articulos")
)
df_agg_dict.show(5, truncate=False)

+----------------------------------------------------------------+-------------+------------------+
|customer_id                                                     |num_articulos|total_normalizado |
+----------------------------------------------------------------+-------------+------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|2            |0.081322033898305 |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|5            |0.0863559322033897|
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|5            |0.190593220338983 |
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|2            |0.0428474576271185|
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|30           |0.7130508474576257|
+----------------------------------------------------------------+-------------+------------------+
only showing top 5 rows


**Total gastado por cliente, sin agrupar** (función ventana — a diferencia de `groupBy().agg()`, no colapsa filas):

In [45]:
from pyspark.sql.window import Window

window_cliente = Window.partitionBy("customer_id")

In [46]:
from pyspark.sql.functions import sum

df_con_total_cliente = df_transactions.withColumn(
    "total_gastado_cliente",
    sum("price").over(window_cliente)
)
df_con_total_cliente.show(5, truncate=False)

[Stage 57:================================>                      (38 + 26) / 64]

+----------+----------------------------------------------------------------+----------+------------------+----------------+------+----------------+----------------+----------+---------+---------------------+
|t_dat     |customer_id                                                     |article_id|price             |sales_channel_id|canal |categoria_precio|tipo_transaccion|dia_semana|tipo_dia |total_gastado_cliente|
+----------+----------------------------------------------------------------+----------+------------------+----------------+------+----------------+----------------+----------+---------+---------------------+
|2018-09-21|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|583558001 |0.0677796610169491|2               |Tienda|Barato          |Regular         |6         |Laborable|0.12706779661016931  |
|2018-09-21|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|639677008 |0.0254067796610169|2               |Tienda|Barato          |Regular         

`truncate=False` es clave acá: `customer_id` es un hash largo, y con el truncado por defecto de `.show()` (20 caracteres) dos IDs distintos pueden lucir idénticos en pantalla — sin `truncate=False` no podrías confirmar si dos filas realmente pertenecen al mismo cliente o no, que es justo lo que esta función ventana busca mostrar.

**No es la misma forma de llegar al mismo resultado — el resultado tiene otra forma.** Compara tu propio `df_agg_dict` (`groupBy().agg()`) contra `df_con_total_cliente`: para un cliente cualquiera, `df_agg_dict` lo muestra **una sola vez**, con el total ya sumado; `df_con_total_cliente` muestra a ese mismo cliente **una vez por cada transacción suya**, todas con el mismo `total_gastado_cliente` repetido — la ventana no colapsó nada, solo le pegó el agregado de su grupo a cada fila individual. Eso habilita cálculos que `groupBy().agg()` no puede hacer, porque ya perdió el detalle: por ejemplo, `price / total_gastado_cliente` te diría qué porcentaje del gasto total de ese cliente representó cada compra puntual — necesitás el valor individual y el agregado en la misma fila a la vez.

**Advertencia de dominio:** `price` está normalizado por Kaggle a [0, 1] — no representa una moneda real. `sum("price")`/`avg("price")` son agregaciones técnicamente correctas, pero leerlas como "gasto en soles/dólares" sería un error de dominio.

## 3.10 Convertir a RDD y procesar texto (`detail_desc` de `articles.csv`)

**Producto del paso:** conteo de palabras distribuido sobre descripciones de producto, con las 10 más frecuentes, más una exploración con `filter()` y un reto de práctica.

Reutiliza `df_articles` (cargado en 3.3) — aplica el patrón RDD de 2.8 sobre `detail_desc`, las ~105 000 descripciones de producto reales. Se hace por partes, para ver el resultado intermedio de cada operación antes de llegar al conteo completo.

**Paso 1: pasar de DataFrame a RDD.** Toma solo la columna de texto y descarta los valores nulos (no todos los artículos tienen descripción):

In [47]:
rdd = df_articles.select("detail_desc").rdd.map(lambda x: x.detail_desc)
rdd = rdd.filter(lambda texto: texto is not None)  # algunos artículos no tienen descripción

rdd.take(5)

['Jersey top with narrow shoulder straps.',
 'Jersey top with narrow shoulder straps.',
 'Jersey top with narrow shoulder straps.',
 'Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.',
 'Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.']

No te sorprendas si varias de las 5 descripciones salen **idénticas** (en una corrida real, las primeras 3 fueron la misma línea repetida) — H&M cataloga cada combinación de color/talla como un artículo distinto, pero muchos comparten la misma descripción de producto. No es un bug del filtro ni de `.take()`, es así el dato real.

**Paso 2: filtrar descripciones por palabra clave.** Antes de ir al conteo completo, una operación más simple sobre el mismo RDD — quedarte solo con las descripciones que mencionan un material real y frecuente en el catálogo, `"cotton"`:

In [48]:
textos_algodon = rdd.filter(lambda texto: "cotton" in texto.lower())
textos_algodon.take(5)

['Sweatshirt in soft organic cotton with a  press-stud on one shoulder (sizes 12-18 months and 18-24 months without a press-stud). Brushed inside.',
 'Sweatshirt in soft organic cotton with a  press-stud on one shoulder (sizes 12-18 months and 18-24 months without a press-stud). Brushed inside.',
 'Leggings in soft jersey with a wide panel at the waist for best fit over the tummy. The cotton content of the leggings is organic.',
 'Leggings in soft jersey with a wide panel at the waist for best fit over the tummy. The cotton content of the leggings is organic.',
 'Leggings in soft jersey with a wide panel at the waist for best fit over the tummy. The cotton content of the leggings is organic.']

**Paso 3:** `flatMap` + `map` + `reduceByKey` — conteo distribuido completo:

In [49]:
import re
from operator import add

palabras = rdd.flatMap(
    lambda linea: re.sub(r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", linea.lower()).split()
)

pares = palabras.filter(lambda p: p != "").map(lambda palabra: (palabra, 1))
conteo = pares.reduceByKey(add)

conteo.takeOrdered(10, key=lambda x: -x[1])

[('and', 163384),
 ('a', 152403),
 ('with', 150704),
 ('the', 136096),
 ('in', 109801),
 ('at', 80690),
 ('back', 36827),
 ('front', 36316),
 ('soft', 35611),
 ('waist', 34416)]

**Lee tu propio resultado, no solo el número:** vas a ver algo como `('and', 163384), ('a', 152403), ('with', 150704), ('the', 136096), ('in', 109801), ('at', 80690)` dominando el top, y recién a partir de ahí (`back`, `front`, `soft`, `waist`) aparece vocabulario real del catálogo. Es esperado, no un error: un conteo de palabras sin quitar *stopwords* (artículos, preposiciones, conjunciones) siempre lo van a dominar palabras funcionales del idioma, sin importar el dominio del texto — recién después aparece el vocabulario que sí describe el negocio (en moda: `back`/`front`/`waist` son partes de la prenda, no personajes ni lugares). Compara esto contra 2.8: en un texto narrativo (la Biblia usada en S1), después de las mismas stopwords aparecerían nombres propios o palabras temáticas distintas — el contraste depende del dominio del texto, no del algoritmo.

**Reto de práctica** (documenta tus respuestas en 3.11, con evidencia real de tu propia corrida):

In [50]:
# 1. ¿Cuántas veces aparece "cotton" en el conteo distribuido?
conteo.filter(lambda x: x[0] == "cotton").collect()

[('cotton', 34348)]

In [51]:
# 2. Extrae 5 descripciones que mencionen "sustainable" (moda sostenible)
rdd.filter(lambda texto: "sustainable" in texto.lower()).take(5)

['CONSCIOUS EXCLUSIVE. Short Bohemian top in a jacquard-weave Orange Fiber™ and Tencel™ lyocell blend with a pattern in glittery threads. Sweetheart neckline at the top with a silicone trim and integral support panels and grosgrain trims that fasten on the inside for extra support, and a zip at the back. Pleats front and back and 3/4-length balloon sleeves with narrow elastication at the top and cuffs. Organic cotton lining. Orange Fiber™ is made from citrus peel that is a by-product of juice production transformed into a sustainable, high-quality fabric that reduces waste and saves natural resources.',
 'CONSCIOUS EXCLUSIVE. Top in matt satin made from a silk and Tencel™ lyocell blend with wide, tie-top shoulder straps in a contrasting colour and a V-neck. Opening with a covered button at the back and slits in the sides. Lined at the top. The top is dyed in a natural, sustainable way using coffee.',
 'Studio Collection. 70s-inspired, 5-pocket jeans in unwashed cotton denim. Fitted at 

**3.** De las 10 palabras más frecuentes de tu propia corrida, ¿cuántas son stopwords (sin significado de dominio) y en qué posición empieza el vocabulario real del catálogo? Responde en una celda markdown propia, con tu resultado real.

## 3.11 Documentar hallazgos y responder preguntas de reflexión

**Producto del paso:** notebook documentado con celdas markdown explicando cada resultado.

Agrega debajo de cada bloque anterior una breve explicación de qué hiciste y qué observaste — es la base directa de la evidencia técnica para 4.3.1.

**Reflexión técnica breve** (5 a 8 líneas): ¿por qué la evaluación perezosa es útil para procesar datos a escala, y qué riesgo tendría si Spark ejecutara cada transformación de inmediato, apenas se escribe?

_(Responde aquí)_